ACTUACION ESPECIAL DE FISCALIZACION RENDICION DE CUENTAS CONTRACTUAL PLATAFORMA SIA OBSERVA VIGENCIA 2025

IMPORTACION DE LIBRERIAS

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import platform
import distro
import unicodedata
import plotly.express as px

from datetime import datetime

print(f"Las librerías se cargaron correctamente.")
print(f"Versión de Pandas: {pd.__version__}")
print(f"Versión de Numpy: {np.__version__}")
print(f"Sistema Operativo: {platform.system()} {platform.release()}")
print(f"Distribución Linux: {distro.name()} {distro.version()} ({distro.codename()})")

Las librerías se cargaron correctamente.
Versión de Pandas: 3.0.0
Versión de Numpy: 2.4.2
Sistema Operativo: Linux 6.14.0-37-generic
Distribución Linux: Ubuntu 24.04 (noble)


CONFIGURACIONES DE VISUALIZACION

In [12]:
pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: "%.2f" % x)
pd.set_option("display.max_info_columns", 500)
plt.style.use("ggplot")
print("configuraciones aplicadas")

configuraciones aplicadas


GESTION DE RUTAS

In [13]:
BASE_DIR = os.path.dirname(os.getcwd())
RAW_DATA_PATH = os.path.join(BASE_DIR, "data", "raw")

PROCESSED_DATA_PATH = os.path.join(BASE_DIR, "data", "processed")

InformeBasico = os.path.join(RAW_DATA_PATH, "Informe_Basico.xlsx")
InformeExtendido = os.path.join(RAW_DATA_PATH, "Informe_Extendido.xlsx")

try:
    df_basico = pd.read_excel(InformeBasico)
    df_extendido = pd.read_excel(InformeExtendido)
    print(f"proyecto ubicado en:{BASE_DIR}")
    print(
        f"Se carga archivo Informe Basico contiene {len(df_basico)} filas e Informe Extendido contiene {len(df_extendido)} filas"
    )

except Exception as e:

    print(f"Error {e}")

proyecto ubicado en:/home/wilo/Escritorio/auditoria_sia
Se carga archivo Informe Basico contiene 14199 filas e Informe Extendido contiene 14138 filas


LIMPIEZA DE CABECERAS

In [14]:
def normalizar_cabeceras(df):
    def limpiar_texto(texto):
        if not isinstance(texto, str):
            return texto

        texto = texto.replace("Ñ", "NH").replace("ñ", "nh")
        texto = (
            unicodedata.normalize("NFKD", texto)
            .encode("ascii", "ignore")
            .decode("utf-8")
        )

        return (
            texto.replace("NH", "N").strip().upper().replace(" ", "_").replace(".", "_")
        )

    df.columns = [limpiar_texto(col) for col in df.columns]
    return df


df_basico = normalizar_cabeceras(df_basico)
df_extendido = normalizar_cabeceras(df_extendido)

print("PROCESANDO INFORME BASICO")
df_basico["VIGENCIA"] = pd.to_numeric(df_basico["VIGENCIA"], errors="coerce").astype(
    "Int64"
)

cols_fecha_basico = [
    "FECHA_SUSCRIPCION",
    "FECHA_ACTA_DE_INICIO",
    "FECHA_TERMINACION",
    "FECHA_CREACION",
    "FECHA_TERMINACION_AMPLIADA",
]

for col in cols_fecha_basico:
    df_basico[col] = pd.to_datetime(
        df_basico[col].astype(str).str.strip().str[:10],
        format="%Y/%m/%d",
        errors="coerce",
    )

print("PROCESANDO INFORME EXTENDIDO")

df_extendido["VIGENCIA"] = pd.to_numeric(
    df_extendido["VIGENCIA"], errors="coerce"
).astype("Int64")
df_extendido["APROPIACION_INICIAL"] = pd.to_numeric(
    df_extendido["APROPIACION_INICIAL"], errors="coerce"
)

cols_fecha_ext = [
    "FECHA_SUSCRIPCION",
    "FECHA_ACTA_DE_INICIO",
    "FECHA_TERMINACION",
    "FECHA_CREACION",
]

for col in cols_fecha_ext:
    df_extendido[col] = pd.to_datetime(
        df_extendido[col].astype(str).str.strip().str[:10],
        format="%Y/%m/%d",
        errors="coerce",
    )

print("\n PROCESAMIENTO EXITOSO")
print(f"Básico: {df_basico.shape} | Extendido: {df_extendido.shape}")

PROCESANDO INFORME BASICO
PROCESANDO INFORME EXTENDIDO

 PROCESAMIENTO EXITOSO
Básico: (14199, 22) | Extendido: (14138, 29)


ENCABEZADOS Y ESTRUCTURA

In [15]:
def inspeccionar_dataframe(df, nombre):
    print(f"\n{'='*20} {nombre} {'='*20}")
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

    resumen = pd.DataFrame(
        {
            "Columna": df.columns,
            "Tipo": df.dtypes.values,
            "No Nulos": df.notnull().sum().values,
            "Nulos": df.isnull().sum().values,
            "Ejemplo": [df[c].iloc[0] if len(df) > 0 else "N/A" for c in df.columns],
        }
    )

    display(resumen)

    print(f"\nVista previa de datos:")
    display(df.head(3))


inspeccionar_dataframe(df_basico, "INFORME BÁSICO")
inspeccionar_dataframe(df_extendido, "INFORME EXTENDIDO")


==================== INFORME BÁSICO ====================
Filas: 14199 | Columnas: 22


,Columna,Tipo,No Nulos,Nulos,Ejemplo
0,TIPO_DE_ENTIDAD,str,14199,0,SOCIEDADES DE ECONOMIA MIXTA
1,NIT,int64,14199,0,901685734
2,ENTIDAD,str,14199,0,UMBRALED S.A.S E.S.P.
3,VIGENCIA,Int64,14126,73,2025
4,CODIGO_CONTRATO,object,14198,1,UMB-01-2025
5,VALOR_INICIAL_CONTRATO,float64,14199,0,17676000.00
6,ADICIONES,float64,14199,0,0.00
7,LIBERACIONES,float64,14199,0,0.00
8,VALOR_VIGENTE,float64,14199,0,17676000.00
9,FECHA_SUSCRIPCION,datetime64[us],14199,0,2025-01-01 00:00:00



Vista previa de datos:


,TIPO_DE_ENTIDAD,NIT,ENTIDAD,VIGENCIA,CODIGO_CONTRATO,VALOR_INICIAL_CONTRATO,ADICIONES,LIBERACIONES,VALOR_VIGENTE,FECHA_SUSCRIPCION,FECHA_ACTA_DE_INICIO,FECHA_TERMINACION,TIEMPO_EJECUCION,MODALIDAD_CONTRATACION,CAUSAL_CONTRATO,TIPO_CONTRATO,FECHA_CREACION,FECHA_TERMINACION_AMPLIADA,NIT_1,NOMBRE,TIPO,ESTADO_CONTRATO
0,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-01-2025,17676000.00,0.00,0.00,17676000.00,2025-01-01,2025-01-01,2025-12-31,364,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,2025-01-31,NaT,24547202,MARIA LILIANA MONTES HOYOS,Contratista,Rendido
1,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-02-2025,8846800.00,0.00,0.00,8846800.00,2025-01-01,2025-01-01,2025-12-31,364,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,2025-01-31,NaT,900010878,GRUPO METRO Y CIA. LTDA,Contratista,Rendido
2,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-03-2025,22176000.00,0.00,0.00,22176000.00,2025-01-02,2025-01-02,2025-12-31,363,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,2025-01-31,NaT,1088290219,DIANA MARCELA RIOS AGUIRRE,Contratista,Rendido



==================== INFORME EXTENDIDO ====================
Filas: 14138 | Columnas: 29


,Columna,Tipo,No Nulos,Nulos,Ejemplo
0,TIPO_DE_ENTIDAD,str,14138,0,SOCIEDADES DE ECONOMIA MIXTA
1,NIT,int64,14138,0,901685734
2,ENTIDAD,str,14138,0,UMBRALED S.A.S E.S.P.
3,VIGENCIA,Int64,14069,69,2025
4,CODIGO_CONTRATO,object,14137,1,UMB-01-2025
5,OBJETO_CONTRATO,str,14138,0,ARRENDAMIENTO DE INMUEBLE CON DESTINACION COMERCIAL
6,FECHA_SUSCRIPCION,datetime64[us],14138,0,2025-01-01 00:00:00
7,FECHA_ACTA_DE_INICIO,datetime64[us],14138,0,2025-01-01 00:00:00
8,FECHA_TERMINACION,datetime64[us],14138,0,2025-12-31 00:00:00
9,TIEMPO_EJECUCION,int64,14138,0,364



Vista previa de datos:


,TIPO_DE_ENTIDAD,NIT,ENTIDAD,VIGENCIA,CODIGO_CONTRATO,OBJETO_CONTRATO,FECHA_SUSCRIPCION,FECHA_ACTA_DE_INICIO,FECHA_TERMINACION,TIEMPO_EJECUCION,VALOR_INICIAL_CONTRATO,ADICIONES,LIBERACIONES,VALOR_VIGENTE,MODALIDAD_CONTRATACION,CAUSAL_CONTRATO,TIPO_CONTRATO,NIT_1,NOMBRE,TIPO,NIT_2,NOMBRE_1,TIPO_1,NOMBRE_DEL_RUBRO,APROPIACION_INICIAL,ORIGEN_RECURSOS,CDPS,RPS,FECHA_CREACION
0,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-01-2025,ARRENDAMIENTO DE INMUEBLE CON DESTINACION COMERCIAL,2025-01-01,2025-01-01,2025-12-31,364,17676000.00,0.00,0.00,17676000.00,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,24547202,MARIA LILIANA MONTES HOYOS,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,SERVICIOS FINANCIEROS Y SERVICIOS CONEXOS; SERVICIOS INMOBILIARIOS; Y SERVICIOS DE ARRENDAMIENTO Y LEASING,68266800.00,Recursos Propios,20250001,20250001,2025-01-31
1,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-02-2025,"CONTRATAR LA IMPLEMENTACION, PUESTA EN FUNCIONAMIENTO Y MANTENIMIENTO DEL SOFTWARE AIRE, BAJO LA MODALIDAD DE ARRENDAMIENTO, QUE CONTENGA LOS MODULOS DE PRESUPUESTO, TESORERIA Y BANCOS, VENTANILLAS DE ATENCION AL PUBLICO, NOMINA Y TALENTO HUMANO, ALMACEN E INVENTARIOS, CONTABILIDAD Y SERVICIOS PUBLICOS DE ENERGIA.",2025-01-01,2025-01-01,2025-12-31,364,8846800.00,0.00,0.00,8846800.00,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,900010878,GRUPO METRO Y CIA. LTDA,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,PAQUETES DE SOFTWARE,9072000.00,Recursos Propios,20250003,20250003,2025-01-31
2,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-03-2025,PRESTAR EL SERVICIO PROFESIONAL EN EL CARGO DE DIRECCION ADMINISTRATIVA Y FINANCIERA PARA LA SOCIEDAD UMBRALED S.A.S. E.S.P.,2025-01-02,2025-01-02,2025-12-31,363,22176000.00,0.00,0.00,22176000.00,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,1088290219,DIANA MARCELA RIOS AGUIRRE,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,SERVICIOS PRESTADOS A LAS EMPRESAS Y SERVICIOS DE PRODUCCION,171903986.00,Recursos Propios,20250005,20250003,2025-01-31


In [16]:
# --- CHEQUEO PARA EL BÁSICO ---
errores_basico = df_basico[
    df_basico["FECHA_TERMINACION"] < df_basico["FECHA_ACTA_DE_INICIO"]
]
df_basico["DURACION_REAL"] = (
    df_basico["FECHA_TERMINACION"] - df_basico["FECHA_ACTA_DE_INICIO"]
).dt.days

# --- CHEQUEO PARA EL EXTENDIDO ---
errores_ext = df_extendido[
    df_extendido["FECHA_TERMINACION"] < df_extendido["FECHA_ACTA_DE_INICIO"]
]
df_extendido["DURACION_REAL"] = (
    df_extendido["FECHA_TERMINACION"] - df_extendido["FECHA_ACTA_DE_INICIO"]
).dt.days

print(f"🚩 BÁSICO: {len(errores_basico)} errores encontrados.")
print(f"🚩 EXTENDIDO: {len(errores_ext)} errores encontrados.")

🚩 BÁSICO: 0 errores encontrados.
🚩 EXTENDIDO: 0 errores encontrados.


ANALISIS DESCRIPTIVO

En este bloque se realiza la consolidación de las cifras reportadas por los 53 sujetos de control en la plataforma SIA OBSERVA para la vigencia 2025, centrando el análisis en dos métricas fundamentales: el conteo de contratos para determinar la cantidad de registros individuales rendidos por cada entidad y la suma del valor vigente, la cual totaliza la inversión real incluyendo el valor inicial, las adiciones y las liberaciones de recursos. El listado resultante se presenta ordenado de mayor a menor según el presupuesto ejecutado para identificar rápidamente las entidades con mayor peso financiero en la contratación, aplicando un filtro de auditoría que incluye únicamente los contratos en estado RENDIDO o RENDIDO EXTEMPORÁNEA y omitiendo aquellos registros en estado REGISTRADO, VERIFICADO o EN PROCESO DE MODIFICACIÓN, garantizando así que el estudio se base solo en información oficial y definitiva.

In [18]:
# 1. Definimos las funciones de formato (Ya las tienes claras)
def formato_moneda_co(valor):
    return f"$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def formato_miles_co(valor):
    return f"{valor:,.0f}".replace(",", ".")

# 2. Filtramos (Asegúrate de que df_basico exista en tu memoria)
estados_validos = ["RENDIDO", "RENDIDO EXTEMPORANEO"]
df_basico_filtrado = df_basico[
    df_basico["ESTADO_CONTRATO"]
    .astype(str)
    .str.upper()
    .str.strip()
    .isin(estados_validos)
].copy()

# 3. Agrupamos por Entidad
col_valor_basico = "VALOR_VIGENTE"
analisis_basico = df_basico_filtrado.groupby("ENTIDAD")[col_valor_basico].agg(["count", "sum"])
analisis_basico.columns = ["NUMERO_CONTRATOS", "TOTAL_VALOR_BASICO"]

# 4. Ordenamos (Aquí estaba el error: corregimos el sort_values)
analisis_basico = analisis_basico.sort_values(
    by="TOTAL_VALOR_BASICO", 
    ascending=False  # <--- Corregido: nada de 'ascendinx' ni coordenadas
).reset_index()

# Re-generamos el índice de Ranking
analisis_basico.index = analisis_basico.index + 1
analisis_basico.index.name = "ITEM"

# 5. Visualización del reporte
print("--- INFORME BÁSICO: CONTRATOS RENDIDOS VIGENCIA 2025 ---")
display(
    analisis_basico.style.format(
        {"NUMERO_CONTRATOS": formato_miles_co, "TOTAL_VALOR_BASICO": formato_moneda_co}
    ).set_properties(**{"text-align": "left"}, subset=["ENTIDAD"])
)

# 6. Resumen rápido
total_contratos = analisis_basico["NUMERO_CONTRATOS"].sum()
total_presupuesto = analisis_basico["TOTAL_VALOR_BASICO"].sum()

print(f"\n📊 RESUMEN GENERAL:")
print(f"- Gran Total Básico: {formato_miles_co(total_contratos)} contratos.")
print(f"- Total presupuesto rendido: {formato_moneda_co(total_presupuesto)}")

--- INFORME BÁSICO: CONTRATOS RENDIDOS VIGENCIA 2025 ---


,ENTIDAD,NUMERO_CONTRATOS,TOTAL_VALOR_BASICO
ITEM,,,
1,GOBERNACIÓN DE RISARALDA,3.793,"$ 476.002.998.528,35"
2,EMPRESA TERRITORIAL DE DESARROLLO URBANO Y RURAL DE RISARALDA,396,"$ 171.631.847.052,00"
3,HOSPITAL SAN JORGE DE PEREIRA,1.089,"$ 169.863.209.719,73"
4,HOSPITAL SANTA MÓNICA,551,"$ 76.836.048.763,00"
5,ALCALDÍA DE SANTA ROSA DE CABAL,1.017,"$ 43.507.767.549,97"
6,HOSPITAL SAN PEDRO Y SAN PABLO LA VIRGINIA,333,"$ 29.756.312.199,00"
7,ALCALDÍA DE PUEBLO RICO,459,"$ 29.127.970.892,94"
8,HOSPITAL SAN VICENTE DE PAUL DE SANTA ROSA DE CABAL,386,"$ 22.686.370.229,00"
9,ALCALDÍA DE MISTRATÓ,411,"$ 18.021.949.123,21"



📊 RESUMEN GENERAL:
- Gran Total Básico: 14.138 contratos.
- Total presupuesto rendido: $ 1.329.650.881.314,29


MODALIDAD DE CONTRATACION

### **Participación Presupuestal por Modalidad**

El siguiente gráfico de anillo (Donut Chart) ilustra la composición porcentual del presupuesto ejecutado en la vigencia 2025.

* **Visión de Proporción:** Permite identificar rápidamente qué porcentaje del "pastel" presupuestal corresponde a procesos competitivos frente a contrataciones directas.
* **Simplificación de Datos:** Se han agrupado las modalidades con menor impacto financiero bajo la categoría **"OTRAS"**, facilitando la lectura de las tendencias dominantes sin sacrificar la integridad del gran total.

### **Tabla Consolidada de Modalidades**

A continuación, se detalla el volumen y la cuantía de la contratación rendida, permitiendo contrastar la frecuencia de los contratos frente a su impacto presupuestal:

* **Relación Volumen/Cuantía:** Esta tabla permite identificar si pocas contrataciones (ej. Licitaciones) mueven más recursos que una gran cantidad de procesos menores (ej. Mínima Cuantía).
* **Base de Auditoría:** Estos totales sirven como punto de control para la verificación de saldos frente a los registros presupuestales de la entidad.

In [19]:
import plotly.express as px
import pandas as pd

# --- PARTE 1: GRÁFICO DE ANILLO (TU SCRIPT) ---
data_donut = (
    df_basico_filtrado.groupby("MODALIDAD_CONTRATACION")["VALOR_VIGENTE"]
    .sum()
    .reset_index()
)
data_donut = data_donut.sort_values(by="VALOR_VIGENTE", ascending=False)

top_n = 6
if len(data_donut) > top_n:
    top_data = data_donut.iloc[:top_n].copy()
    others_val = data_donut.iloc[top_n:]["VALOR_VIGENTE"].sum()
    others_row = pd.DataFrame(
        [{"MODALIDAD_CONTRATACION": "OTRAS", "VALOR_VIGENTE": others_val}]
    )
    data_donut = pd.concat([top_data, others_row], ignore_index=True)

fig_donut = px.pie(
    data_donut,
    values="VALOR_VIGENTE",  # Reestablecí values para que el % sea real
    names="MODALIDAD_CONTRATACION",
    hole=0.5,
    title="Distribución Presupuestal por Modalidad (Anillo)",
    color_discrete_sequence=px.colors.qualitative.Pastel,
)

fig_donut.update_traces(
    textinfo="percent",
    textposition="inside",
    texttemplate="%{percent:.0%}",
    hoverinfo="none",
)

fig_donut.update_layout(
    hovermode=False,
    legend_title="Modalidades",
    annotations=[dict(text="TOTAL", x=0.5, y=0.5, font_size=20, showarrow=False)],
)

fig_donut.show()

# --- PARTE 2: TABLA RESUMEN (TU SCRIPT CON ALINEACIÓN IZQUIERDA) ---
tabla_resumen = (
    df_basico_filtrado.groupby("MODALIDAD_CONTRATACION")["VALOR_VIGENTE"]
    .agg(["count", "sum"])
    .reset_index()
)

tabla_resumen.columns = [
    "MODALIDAD DE CONTRATACIÓN",
    "CANTIDAD DE CONTRATOS",
    "CUANTÍA TOTAL",
]
tabla_resumen = tabla_resumen.sort_values(by="CUANTÍA TOTAL", ascending=False)

print("\n" + "─" * 50)
print("📋 RESUMEN DE CONTRATACIÓN POR MODALIDAD (Muestra Rendida)")
print("─" * 50)

# Mostramos la tabla alineada a la izquierda para que se vea melita
display(
    tabla_resumen.style.format(
        {"CANTIDAD DE CONTRATOS": formato_miles_co, "CUANTÍA TOTAL": formato_moneda_co}
    )
    .set_properties(
        **{"text-align": "left"}  # Aquí es donde se alinea a la izquierda, parce
    )
    .hide(axis="index")
)


──────────────────────────────────────────────────
📋 RESUMEN DE CONTRATACIÓN POR MODALIDAD (Muestra Rendida)
──────────────────────────────────────────────────


MODALIDAD DE CONTRATACIÓN,CANTIDAD DE CONTRATOS,CUANTÍA TOTAL
CONTRATACIÓN DIRECTA,7.946,"$ 451.550.951.053,59"
SIN OFERTAS,3.786,"$ 210.060.318.209,13"
CON OFERTAS,115,"$ 202.258.828.101,43"
LICITACIONES PÚBLICAS,25,"$ 158.695.849.719,00"
RÉGIMEN ESPECIAL,1.047,"$ 131.073.531.934,73"
SELECCIÓN ABREVIADA,129,"$ 91.458.854.926,33"
INVITACIÓN DIRECTA,301,"$ 17.149.950.947,90"
MÍNIMA CUANTÍA,564,"$ 14.967.826.603,52"
CONVOCATORIA PÚBLICA - DECRETO 092/2017,10,"$ 12.611.254.873,00"
ACUERDOS MARCO DE PRECIOS,66,"$ 12.012.621.291,45"


In [20]:
import plotly.express as px
import pandas as pd

# --- PARTE 1: PREPARACIÓN DE DATOS ---
df_causal = (
    df_basico_filtrado.groupby("CAUSAL_CONTRATO")["VALOR_VIGENTE"]
    .agg(["count", "sum"])
    .reset_index()
)
df_causal.columns = ["CAUSAL", "CANTIDAD", "CUANTIA"]
df_causal = df_causal.sort_values(by="CUANTIA", ascending=False)

# Agrupamos las causales pequeñas para que el gráfico no sea un caos
top_n_causales = 10
if len(df_causal) > top_n_causales:
    df_top = df_causal.iloc[:top_n_causales].copy()
    df_others = pd.DataFrame(
        [
            {
                "CAUSAL": "OTRAS CAUSALES",
                "CANTIDAD": df_causal.iloc[top_n_causales:]["CANTIDAD"].sum(),
                "CUANTIA": df_causal.iloc[top_n_causales:]["CUANTIA"].sum(),
            }
        ]
    )
    df_grafico_causal = pd.concat([df_top, df_others], ignore_index=True)
else:
    df_grafico_causal = df_causal

# --- PARTE 2: GRÁFICO TREEMAP (SIN HOVER) ---
fig_causal = px.treemap(
    df_grafico_causal,
    path=["CAUSAL"],  # La jerarquía
    values="CUANTIA",
    title="Distribución Presupuestal por Causal de Contratación",
    color="CUANTIA",
    color_continuous_scale="Blues",  # Escala de azules profesional
)

fig_causal.update_traces(
    textinfo="label+percent root",  # Muestra el nombre y el % del total
    texttemplate="<b>%{label}</b><br>%{percentRoot:.0%}",  # Negrita y sin decimales
    hoverinfo="none",
)

fig_causal.update_layout(hovermode=False)
fig_causal.show()

# --- PARTE 3: TABLA RESUMEN ALINEADA ---
print("\n" + "─" * 50)
print("📋 DETALLE POR CAUSAL DE CONTRATACIÓN")
print("─" * 50)

display(
    df_causal.style.format({"CANTIDAD": formato_miles_co, "CUANTIA": formato_moneda_co})
    .set_properties(**{"text-align": "left"})
    .hide(axis="index")
)


──────────────────────────────────────────────────
📋 DETALLE POR CAUSAL DE CONTRATACIÓN
──────────────────────────────────────────────────


CAUSAL,CANTIDAD,CUANTIA
Manual de contratacion,3.901,"$ 412.319.146.310,56"
PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,7.840,"$ 258.803.729.517,64"
LICITACIÓN PÚBLICA,25,"$ 167.070.873.689,00"
CONTRATOS INTERADMINISTRATIVOS,151,"$ 111.220.618.378,97"
CONVENIOS INTERADMINISTRATIVOS,161,"$ 74.193.235.494,18"
SUBASTA INVERSA,69,"$ 71.612.600.070,81"
PRESTACIÓN DE SERVICIOS DE SALUD,197,"$ 68.635.614.331,73"
MANUAL DE CONTRATACIÓN,600,"$ 43.925.843.122,14"
URGENCIA MANIFIESTA,6,"$ 23.309.162.410,00"
Mínima Cuantía,564,"$ 14.967.826.603,52"
